# 03 - Baselines

Before fitting anything clever, set the bar with models that know almost nothing.
Anything later has to beat these on the same data.

**Target:** `delta_s`, a lap's time minus the driver's own median clean lap in that
race (see `f1lab/features.py`). This removes car, driver and track pace, leaving tyre,
fuel and conditions.

**Evaluation:** train on 2022-2023, score on 2024. The 2025 season is held back for
one final evaluation at the end and is not touched here. Five wet or drying races are
excluded by a pace-based rule applied to every season.

In [1]:
import numpy as np
import pandas as pd
from f1lab.features import prepare, split, mae, rmse, drying_races

raw = pd.read_parquet("../data/laps.parquet")
clean = pd.read_parquet("../data/laps_clean.parquet")
print("excluded wet/drying races (season, round):", sorted(drying_races(clean)))
df = prepare(raw, clean)
train, val = split(df, "dev")
print(f"{len(df)} laps | train {len(train)} (2022-2023) | validation {len(val)} (2024)")
unseen = ~val["track"].isin(train["track"].unique())
print("validation laps at tracks never seen in training:", int(unseen.sum()), sorted(val[unseen]["track"].unique()))
df["delta_s"].describe(percentiles=[.01, .5, .99]).round(2)

excluded wet/drying races (season, round): [(2022, 7), (2023, 14), (2025, 1), (2025, 12), (2025, 13)]
79124 laps | train 36745 (2022-2023) | validation 21572 (2024)
validation laps at tracks never seen in training: 778 ['Shanghai']


count    79124.00
mean         0.08
std          1.44
min         -9.35
1%          -2.47
50%          0.00
99%          3.65
max         31.53
Name: delta_s, dtype: float64

The target has a heavy tail: it runs from about -9 s to over +31 s. These are mostly
real incident laps (spins, off-track moments, unflagged yellows) from drivers with
plenty of clean laps, not artefacts of a bad base pace. Counting them:

In [2]:
big = df[df["delta_s"].abs() > 8]
print(f"laps with |delta| > 8 s: {len(big)} ({len(big) / len(df):.2%} of all laps)")
print("RMSE is dominated by these; MAE is the headline metric, RMSE secondary.")

laps with |delta| > 8 s: 301 (0.38% of all laps)
RMSE is dominated by these; MAE is the headline metric, RMSE secondary.


## The baselines

In [3]:
y = val["delta_s"]
by_compound = train.groupby("compound")["delta_s"].mean()
by_track_compound = train.groupby(["track", "compound"])["delta_s"].mean()
bins = [0, 5, 10, 15, 20, 30, 100]
by_age = train.assign(b=pd.cut(train["tyre_age"], bins)).groupby(["compound", "b"], observed=True)["delta_s"].mean()
val_b = val.assign(b=pd.cut(val["tyre_age"], bins))

preds = {
    "B0 predict 0 (no change)": pd.Series(0.0, index=val.index),
    "B1 mean by compound": val["compound"].map(by_compound),
    "B2 mean by track + compound": pd.Series(
        [by_track_compound.get((t, c), np.nan) for t, c in zip(val["track"], val["compound"])], index=val.index
    ).fillna(val["compound"].map(by_compound)),
    "B3 mean by compound + tyre-age bucket": pd.Series(
        [by_age.get((c, b), np.nan) for c, b in zip(val_b["compound"], val_b["b"])], index=val.index
    ).fillna(val["compound"].map(by_compound)),
}
results = pd.DataFrame({k: {"MAE (s)": mae(y, p), "RMSE (s)": rmse(y, p)} for k, p in preds.items()}).T.round(3)
results

,MAE (s),RMSE (s)
B0 predict 0 (no change),0.908,1.429
B1 mean by compound,0.883,1.409
B2 mean by track + compound,0.891,1.420
B3 mean by compound + tyre-age bucket,0.882,1.407


Reading the table:
- Predicting no change from the driver's median pace is off by about 0.91 s on average.
- Compound averages (B1) and compound-by-tyre-age averages (B3) only trim that to
  about 0.88 s. Tyre age adds almost nothing over compound alone.
- Adding the track (B2) is slightly *worse* than compound alone (0.891 vs 0.883).
  Per-track averages are too noisy to help when trained on two seasons.
- The per-compound means are odd: MEDIUM is +0.31 s but HARD and SOFT are about 0. A
  likely reason, not yet tested, is *when* each compound is used. Mediums run early on
  heavy fuel, hards later on light fuel, so the compound is partly standing in for fuel
  load.

None of these baselines knows the lap number, and fuel burn moves lap time by
seconds across a race, so the obvious next step is a model with a fuel term. That is
the hypothesis the linear model will test.

In [4]:
print("training mean delta by compound:", by_compound.round(3).to_dict())
print(f"spread of the target on validation: mean abs deviation {(y - y.median()).abs().mean():.3f} s, std {y.std():.3f} s")

training mean delta by compound: {'HARD': -0.02, 'MEDIUM': 0.306, 'SOFT': -0.081}
spread of the target on validation: mean abs deviation 0.908 s, std 1.428 s
